# Fisher Matrix Exploration Notebook

This notebook loads a single microlensing event and its posterior samples to experiment with Fisher matrix stuff. Fill in the paths below and run to load your data.

## 1. Set File Paths

Fill in these paths with your specific event data:

In [ ]:
# ==== FILL IN THESE PATHS ====

# Path to the data directory (e.g., Fisher_overguide_m40)
data_path = "/Users/malpas.1/Code/GullsPosteriors/Fisher_overguide_m40/"

# Specific lightcurve file to load (.det.lc file)
lightcurve_file = ""  # e.g., "5f_overguide_m40_1_721_321.det.lc"

# Posterior samples file (.npy file) - leave empty if not yet generated
samples_file = ""  # e.g., "posteriors/721_1_321_post_samples.npy"

# Event name for plots
event_name = ""  # e.g., "721_1_321"

# =================================

import os
print(f"Data path: {data_path}")
print(f"Lightcurve file: {lightcurve_file}")
print(f"Samples file: {samples_file}")
print(f"Event name: {event_name}")

## 2. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import corner
import pickle
import warnings
warnings.filterwarnings('ignore')

# Import the GullsPosteriors modules
from Data import Data
from Parallax import Parallax
from Event import Event
from Fit import Fit
from Orbit import Orbit

print("Libraries imported successfully!")

## Fisher Matrix Parameters: Linear vs Log Space

Fisher matrix uncertainties are computed for the following parameters, with their respective parameterization spaces:

| Parameter | Space        | Description                    |
|-----------|--------------|--------------------------------|
| t0        | Linear       | Time of maximum magnification  |
| tE        | Logarithmic  | Einstein radius crossing time  |
| u0        | Linear       | Impact parameter               |
| alpha     | Linear       | Source trajectory angle        |
| s         | Logarithmic  | Projected separation           |
| q         | Logarithmic  | Mass ratio                     |
| rho       | Logarithmic  | Finite source effect           |
| piEN      | Linear       | Parallax (North component)     |
| piEE      | Linear       | Parallax (East component)      |
| Fb        | Linear       | Baseline flux                  |
| fs        | Linear       | Source flux                    |

**Note:** Flux parameters Fb and fs are computed individually for each filter.

## 3. Load and Inspect Data

Load the lightcurve data and convert it to a DataFrame for easy exploration:

In [ ]:
# Initialize the Data object
data_obj = Data()

if lightcurve_file:
    # Load the specific lightcurve file
    full_lc_path = os.path.join(data_path, lightcurve_file)
    
    if os.path.exists(full_lc_path):
        print(f"Loading lightcurve from: {full_lc_path}")
        
        # Load the data using the Data class method
        data_dict = data_obj.load_data(full_lc_path)
        
        print(f"Found data for {len(data_dict)} observatories: {list(data_dict.keys())}")
        
        # Show the Fisher matrix information if available
        if hasattr(data_obj, 'model_derivatives') and data_obj.model_derivatives is not None:
            print(f"Fisher derivatives shape: {data_obj.model_derivatives.shape}")
            print(f"Fisher covariance available: {hasattr(data_obj, 'model_covariance')}")
            if hasattr(data_obj, 'model_parameter_uncertainties'):
                print(f"Parameter uncertainties available: {len(data_obj.model_parameter_uncertainties)} parameters")
        else:
            print("No Fisher derivative information found in this file")
    else:
        print(f"Lightcurve file not found: {full_lc_path}")
        print("Available files in directory:")
        for f in os.listdir(data_path):
            if f.endswith('.det.lc'):
                print(f"  {f}")
else:
    print("Please specify a lightcurve file in the first cell")

### Convert to DataFrame for Easy Exploration

In [ ]:
if 'data_dict' in locals():
    # Column names based on the load_data method documentation
    column_names = [
        'BJD',
        'measured_relative_flux', 
        'measured_relative_flux_error',
        'parallax_shift_t',
        'parallax_shift_u', 
        'true_relative_flux',
        'true_relative_flux_error',
        'Simulation_time'
    ]
    
    # Create DataFrames for each observatory
    dfs = {}
    for obs_code, obs_data in data_dict.items():
        # obs_data is shape (8, n_points), so transpose to get (n_points, 8)
        df = pd.DataFrame(obs_data.T, columns=column_names)
        df['observatory_code'] = obs_code
        dfs[obs_code] = df
        
        print(f"Observatory {obs_code}: {len(df)} data points")
        print(f"  Time range: {df['BJD'].min():.1f} to {df['BJD'].max():.1f}")
        print(f"  Flux range: {df['measured_relative_flux'].min():.3f} to {df['measured_relative_flux'].max():.3f}")
        print()
    
    # Combine all observatories into one DataFrame
    combined_df = pd.concat(dfs.values(), ignore_index=True)
    print(f"Combined DataFrame: {len(combined_df)} total data points")
    
    # Show the first few rows
    print("\nFirst 5 rows:")
    display(combined_df.head())
else:
    print("No data loaded yet. Please load lightcurve data first.")

## 4. Data Processing and Analysis

Load posterior samples if available and analyze the parameter distributions:

In [ ]:
# Load posterior samples if available
samples = None
truths = None

if samples_file and os.path.exists(os.path.join(data_path, samples_file)):
    samples_path = os.path.join(data_path, samples_file)
    print(f"Loading samples from: {samples_path}")
    samples = np.load(samples_path)
    print(f"Samples shape: {samples.shape} (n_samples, n_parameters)")
    
    # Try to load corresponding truths file
    truths_file = samples_file.replace('_post_samples.npy', 'end_truths.pkl')
    truths_path = os.path.join(data_path, truths_file)
    
    if os.path.exists(truths_path):
        print(f"Loading truths from: {truths_path}")
        with open(truths_path, 'rb') as f:
            truths = pickle.load(f)
        print(f"Truths loaded: {len(truths)} entries")
    else:
        print(f"Truths file not found: {truths_path}")
        
elif samples_file:
    print(f"Samples file not found: {os.path.join(data_path, samples_file)}")
else:
    print("No samples file specified. This is fine if you haven't run the sampler yet.")
    print("You can still explore the lightcurve data and Fisher information.")

### Fisher Matrix Analysis

In [ ]:
# Analyze Fisher matrix if available
if hasattr(data_obj, 'model_covariance') and data_obj.model_covariance is not None:
    fisher_cov = data_obj.model_covariance
    fisher_unc = data_obj.model_parameter_uncertainties
    
    print(f"Fisher covariance matrix shape: {fisher_cov.shape}")
    print(f"Fisher uncertainties: {fisher_unc}")
    
    # Parameter names (in the order used by the code)
    param_names = ['s', 'q', 'rho', 'u0', 'alpha', 't0', 'tE', 'piEE', 'piEN']
    
    # Create DataFrame for Fisher covariance
    fisher_df = pd.DataFrame(fisher_cov, index=param_names, columns=param_names)
    print("\nFisher covariance matrix:")
    display(fisher_df)
    
    # Calculate correlation matrix
    fisher_corr = fisher_cov / np.outer(fisher_unc, fisher_unc)
    corr_df = pd.DataFrame(fisher_corr, index=param_names, columns=param_names)
    print("\nFisher correlation matrix:")
    display(corr_df)
    
else:
    print("No Fisher matrix information available.")
    print("This might be because:")
    print("1. The lightcurve file doesn't contain Fisher derivative columns")
    print("2. The Fisher computation failed")
    print("3. You haven't loaded a lightcurve file yet")

## 5. Visualization

Create plots to visualize the data and posterior samples:

### Plot Lightcurve Data

In [ ]:
if 'combined_df' in locals():
    plt.figure(figsize=(12, 8))
    
    # Colors for different observatories
    colors = {0: 'orange', 1: 'red', 2: 'green'}
    labels = {0: 'W146', 1: 'Z087', 2: 'K213'}
    
    for obs_code in combined_df['observatory_code'].unique():
        obs_data = combined_df[combined_df['observatory_code'] == obs_code]
        
        plt.subplot(2, 1, 1)
        plt.errorbar(obs_data['BJD'], obs_data['measured_relative_flux'], 
                    yerr=obs_data['measured_relative_flux_error'],
                    fmt='.', color=colors.get(obs_code, 'blue'), 
                    label=labels.get(obs_code, f'Obs {obs_code}'), alpha=0.7)
        
        plt.subplot(2, 1, 2)
        # Plot magnification (assuming baseline flux is minimum)
        baseline = obs_data['measured_relative_flux'].min()
        magnification = obs_data['measured_relative_flux'] / baseline
        plt.errorbar(obs_data['BJD'], magnification,
                    yerr=obs_data['measured_relative_flux_error'] / baseline,
                    fmt='.', color=colors.get(obs_code, 'blue'), alpha=0.7)
    
    plt.subplot(2, 1, 1)
    plt.ylabel('Relative Flux')
    plt.legend()
    plt.title(f'Lightcurve Data - {event_name}' if event_name else 'Lightcurve Data')
    
    plt.subplot(2, 1, 2)
    plt.xlabel('BJD')
    plt.ylabel('Magnification')
    plt.yscale('log')
    
    plt.tight_layout()
    plt.show()
else:
    print("No lightcurve data to plot. Please load data first.")

### Corner Plot WITHOUT Fisher Annotations

This creates a clean corner plot without any Fisher matrix overlays:

In [ ]:
if samples is not None:
    # Parameter labels (adjust based on whether LOM is enabled)
    if samples.shape[1] == 12:  # LOM enabled
        labels = [
            r"$s$", r"$q$", r"$\rho$", 
            r"$u_0$", r"$\alpha$", r"$t_0$", r"$t_E$", 
            r"$\pi_{EE}$", r"$\pi_{EN}$", 
            r"$i$", r"$\phi$", r"$P$"
        ]
    else:  # 9 parameters (no LOM)
        labels = [
            r"$s$", r"$q$", r"$\rho$", 
            r"$u_0$", r"$\alpha$", r"$t_0$", r"$t_E$", 
            r"$\pi_{EE}$", r"$\pi_{EN}$"
        ]
    
    # Get truth values if available
    truth_values = None
    if truths is not None and 'params' in truths:
        truth_values = truths['params'][:samples.shape[1]]
    
    print(f"Creating corner plot for {samples.shape[1]} parameters...")
    print(f"Using {len(samples)} samples")
    
    # Create clean corner plot
    fig = corner.corner(
        samples,
        labels=labels,
        truths=truth_values,
        truth_color='red',
        show_titles=True,
        title_kwargs={"fontsize": 12},
        label_kwargs={"fontsize": 14}
    )
    
    plt.suptitle(f'Posterior Samples - {event_name}' if event_name else 'Posterior Samples', 
                fontsize=16, y=0.98)
    plt.show()
    
    print("Clean corner plot created (no Fisher annotations)")
    
else:
    print("No posterior samples available for corner plot.")
    print("Run the sampler first to generate samples, then load them using the paths at the top.")

### Fisher Matrix Visualization

In [ ]:
if hasattr(data_obj, 'model_covariance') and data_obj.model_covariance is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot covariance matrix
    im1 = ax1.imshow(fisher_cov, cmap='viridis')
    ax1.set_title('Fisher Covariance Matrix')
    ax1.set_xticks(range(len(param_names)))
    ax1.set_yticks(range(len(param_names)))
    ax1.set_xticklabels(param_names, rotation=45)
    ax1.set_yticklabels(param_names)
    plt.colorbar(im1, ax=ax1, shrink=0.8)
    
    # Plot correlation matrix
    im2 = ax2.imshow(fisher_corr, cmap='coolwarm', vmin=-1, vmax=1)
    ax2.set_title('Fisher Correlation Matrix')
    ax2.set_xticks(range(len(param_names)))
    ax2.set_yticks(range(len(param_names)))
    ax2.set_xticklabels(param_names, rotation=45)
    ax2.set_yticklabels(param_names)
    plt.colorbar(im2, ax=ax2, shrink=0.8)
    
    plt.tight_layout()
    plt.show()
else:
    print("No Fisher matrix to visualize.")

## 6. Export Results

Save processed data and analysis results:

In [ ]:
# Save the lightcurve DataFrame if you want
if 'combined_df' in locals() and event_name:
    output_file = f"{event_name}_lightcurve_data.csv"
    combined_df.to_csv(output_file, index=False)
    print(f"Lightcurve data saved to: {output_file}")

# Save Fisher matrix information if available
if hasattr(data_obj, 'model_covariance') and data_obj.model_covariance is not None and event_name:
    fisher_output = {
        'covariance': fisher_cov,
        'uncertainties': fisher_unc,
        'correlation': fisher_corr,
        'parameter_names': param_names
    }
    
    fisher_file = f"{event_name}_fisher_info.pkl"
    with open(fisher_file, 'wb') as f:
        pickle.dump(fisher_output, f)
    print(f"Fisher information saved to: {fisher_file}")

print("\nNotebook execution complete!")
print("You can now experiment with the Fisher stuff in the cells above.")